# S2.T1.3 · Made in Rwanda Content Recommender — Evaluation Notebook

**Evaluated on 120 queries** | Results as of 2026-04-22

| Metric | Value |
|---|---|
| NDCG@5 (mean) | **0.0808** |
| Local-presence rate (≥1 local in top-3) | **100.0 %** |
| Curated-fallback rate | **0.0 %** |

This notebook:
1. Loads `catalog.csv`, `queries.csv`, and `click_log.csv` from `data/`
2. Instantiates the TF-IDF recommender from `recommender.py`
3. Builds graded relevance labels from click logs + baseline labels
4. Computes **NDCG@5** and **local-presence rate** across all queries
5. Saves evaluation results to CSV


## 0. Install dependencies (Colab)

In [ ]:
import subprocess, sys
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q',
                       'pandas', 'numpy', 'scikit-learn', 'scipy'])
print('Dependencies ready.')

## 1. Imports

In [ ]:
import numpy as np
import pandas as pd
from pathlib import Path
from sklearn.metrics import ndcg_score

from recommender import MadeInRwandaRecommender

DATA_DIR = Path('data')
print('Data directory:', DATA_DIR.resolve())

## 2. Load data

In [ ]:
catalog   = pd.read_csv(DATA_DIR / 'catalog.csv')
queries   = pd.read_csv(DATA_DIR / 'queries.csv')

click_log_path = DATA_DIR / 'click_log.csv'
click_log = pd.read_csv(click_log_path) if click_log_path.exists() else pd.DataFrame()

print(f'Catalog  : {len(catalog):,} products | columns: {list(catalog.columns)}')
print(f'Queries  : {len(queries):,} rows    | columns: {list(queries.columns)}')
print(f'Click log: {len(click_log):,} events  | columns: {list(click_log.columns)}')

## 3. Instantiate recommender

In [ ]:
rec = MadeInRwandaRecommender(
    data_dir=str(DATA_DIR),
    similarity_threshold=0.18,
    local_boost=0.15,
    popularity_weight=0.05,
)
print(f'Recommender ready. Catalog size: {len(rec.catalog)}')

## 4. Detect query and label columns
Tries common column name variants so the notebook works without manual edits.

In [ ]:
CANDIDATE_QUERY_COLS = ['query', 'search_query', 'q', 'text', 'query_text']
CANDIDATE_LABEL_COLS = ['global_best_match', 'baseline_sku', 'best_match_sku',
                        'label_sku', 'relevant_sku', 'sku']

query_col = next((c for c in CANDIDATE_QUERY_COLS if c in queries.columns), None)
label_col = next((c for c in CANDIDATE_LABEL_COLS if c in queries.columns), None)

if query_col is None:
    raise ValueError(
        f'No query text column found. Available: {list(queries.columns)}\n'
        f'Please rename your query column to one of: {CANDIDATE_QUERY_COLS}'
    )

print(f'Query column : "{query_col}"')
print(f'Label column : "{label_col}" (None = will rely on click log only)')

## 5. Build graded relevance
- **Primary**: click counts from `click_log.csv`, normalised per query.
- **Secondary**: `global_best_match` / baseline label from `queries.csv` (relevance = 1.0).
- Queries with no relevance data yield `NaN` NDCG and are excluded from the mean.

In [ ]:
catalog_skus = rec.catalog['sku'].astype(str).tolist()
sku_to_idx   = {sku: i for i, sku in enumerate(catalog_skus)}
N = len(catalog_skus)

relevance_map: dict = {}

if not click_log.empty:
    q_click_col = next((c for c in CANDIDATE_QUERY_COLS if c in click_log.columns), None)
    q_sku_col   = next((c for c in ['sku', 'product_sku', 'item_sku', 'clicked_sku']
                        if c in click_log.columns), None)

    if q_click_col and q_sku_col:
        tmp = click_log.copy()
        tmp[q_click_col] = tmp[q_click_col].astype(str)
        tmp[q_sku_col]   = tmp[q_sku_col].astype(str)
        grouped = tmp.groupby([q_click_col, q_sku_col]).size().reset_index(name='clicks')

        for q_val, grp in grouped.groupby(q_click_col):
            rel = np.zeros(N, dtype=float)
            max_clicks = grp['clicks'].max()
            for _, row in grp.iterrows():
                sku = row[q_sku_col]
                if sku in sku_to_idx:
                    rel[sku_to_idx[sku]] = row['clicks'] / max_clicks
            relevance_map[q_val] = rel
        print(f'Built relevance from click log for {len(relevance_map)} queries.')
    else:
        print('Click log found but columns not detected. Skipping.')
else:
    print('No click log — relying on label column only.')

if label_col:
    for _, row in queries.iterrows():
        q_val = str(row[query_col])
        sku   = str(row[label_col])
        if q_val not in relevance_map:
            relevance_map[q_val] = np.zeros(N, dtype=float)
        if sku in sku_to_idx:
            relevance_map[q_val][sku_to_idx[sku]] = max(
                relevance_map[q_val][sku_to_idx[sku]], 1.0
            )
    print(f'Merged label column. Total queries with relevance: {len(relevance_map)}')

print(f'Total queries with relevance data: {len(relevance_map)} / {len(queries)}')

## 6. Evaluate NDCG@5 and Local-Presence Rate

**Expected results (from run on 2026-04-22):**
- NDCG@5 = **0.0808**
- Local-presence rate = **100.0 %** (every query had ≥1 local product in top-3)
- Curated-fallback rate = **0.0 %**
- n = **120 queries**

In [ ]:
rows       = []
ndcg_vals  = []
local_hits = []

for _, row in queries.iterrows():
    q_val = str(row[query_col])

    try:
        top5 = rec.recommend(q_val, top_k=5)
    except Exception as e:
        print(f'  [WARN] recommend() failed for "{q_val}": {e}')
        continue

    pred_skus  = top5['sku'].astype(str).tolist()
    top3_local = int(top5.head(3)['is_local'].any())
    local_hits.append(top3_local)

    y_true = relevance_map.get(q_val, np.zeros(N, dtype=float))
    y_score = np.zeros(N, dtype=float)
    for rank, sku in enumerate(reversed(pred_skus), start=1):
        if sku in sku_to_idx:
            y_score[sku_to_idx[sku]] = rank

    ndcg_val = ndcg_score([y_true], [y_score], k=5) if y_true.sum() > 0 else np.nan
    ndcg_vals.append(ndcg_val)

    rows.append({
        'query'          : q_val,
        'top1_sku'       : pred_skus[0] if pred_skus else None,
        'top1_is_local'  : bool(top5.iloc[0]['is_local']) if len(top5) > 0 else None,
        'top3_has_local' : bool(top3_local),
        'ndcg5'          : ndcg_val,
        'reason'         : top5.iloc[0]['reason'] if len(top5) > 0 else None,
    })

print(f'Evaluated {len(rows)} / {len(queries)} queries.')

## 7. Summary metrics

In [ ]:
mean_ndcg      = float(np.nanmean(ndcg_vals))
local_presence = float(np.mean(local_hits)) if local_hits else float('nan')
n_with_rel     = int(np.sum(~np.isnan(ndcg_vals)))
fallback_rate  = sum(1 for r in rows if r['reason'] == 'curated_fallback') / max(len(rows), 1)

summary = pd.DataFrame([
    {'metric': 'NDCG@5 (mean, queries with relevance)',
     'value': f'{mean_ndcg:.4f}',
     'n': n_with_rel},
    {'metric': 'Local-presence rate (≥1 local in top-3)',
     'value': f'{local_presence:.4f}  ({local_presence*100:.1f}%)',
     'n': len(rows)},
    {'metric': 'Curated-fallback rate',
     'value': f'{fallback_rate:.4f}  ({fallback_rate*100:.1f}%)',
     'n': len(rows)},
])

print('='*60)
print(f'  NDCG@5               : {mean_ndcg:.4f}  (expected 0.0808)')
print(f'  Local-presence rate  : {local_presence:.4f}  (expected 1.0000 = 100.0%)')
print(f'  Curated-fallback rate: {fallback_rate:.4f}  (expected 0.0000)')
print('='*60)
summary

## 8. Per-query results table

In [ ]:
results_df = pd.DataFrame(rows)

# Highlight queries with highest NDCG
top_queries = results_df.nlargest(10, 'ndcg5')[['query','top1_sku','top3_has_local','ndcg5','reason']]
print('Top 10 queries by NDCG@5:')
print(top_queries.to_string(index=False))

results_df.head(20)

## 9. Quick demo — required video demo queries

In [ ]:
# These two queries are required in the 4-minute video walkthrough
demo_queries = [
    'leather boots',                   # English example
    'cadeau en cuir pour femme',        # French example — required by brief
]

for q in demo_queries:
    print(f'\n--- Query: "{q}" ---')
    top = rec.recommend(q, top_k=5)
    cols = ['rank','sku','title','origin_district','is_local','base_similarity','score','reason']
    print(top[cols].to_string(index=False))

## 10. Save outputs

Expected output files:
- `evaluation_results.csv` — per-query results (120 rows)
- `evaluation_summary.csv` — aggregated metrics

**Reference values (2026-04-22 run):**
```
NDCG@5 (mean)          = 0.0808
Local-presence rate    = 1.0000  (100.0%)
Curated-fallback rate  = 0.0000  (0.0%)
n queries              = 120
```

In [ ]:
results_df.to_csv('evaluation_results.csv', index=False)
summary.to_csv('evaluation_summary.csv', index=False)
print('Saved: evaluation_results.csv  |  evaluation_summary.csv')
print(f'Total rows saved: {len(results_df)}')